# V5W_01 — Ingest 5words → `data/5words_subjects/`

Legge le epoche EEGLAB (.set, già preprocessate) e le scrive nel formato canonico della tesi: un CSV `(61, 384)` per trial, in `data/5words_subjects/P{id:03d}_S{sess:03d}/{parola}_{cond}_{k}.csv`.

**Env: `daniele_311`** (serve MNE).

In [ ]:
# V5W_01 — Ingest delle epoche 5words nel formato canonico
#
# Le .set sono GIA' preprocessate (band-pass 1-100, notch 50/100, 256 Hz, avg ref,
# pop_clean_rawdata) ed epocate a 61x384. Questo stadio NON ri-filtra: legge le
# epoche EEGLAB e le scrive come CSV (61,384) in data/5words_subjects/, un file per trial.
# Env: daniele_311 (serve MNE).
import warnings; warnings.filterwarnings('ignore')
import re, json
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, mne
from tqdm.auto import tqdm

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
SRC_ROOT = Path('/Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - 5words')
OUT_ROOT = project_root / 'data' / '5words_subjects'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

WORDS  = ['acqua', 'aiuto', 'mangiare', 'no', 'si']
CONDS  = ['img', 'read']
N_CHAN, N_SAMP = 61, 384
TO_MICROVOLT = 1e6   # MNE legge le .set in Volt; la tesi salva in microVolt

assert SRC_ROOT.exists(), f'sorgente non trovata: {SRC_ROOT}'
print('project_root:', project_root)
print('OUT_ROOT    :', OUT_ROOT)


## Ingest

In [ ]:
# Soggetti con epoche pronte
subj_dirs = sorted(d for d in SRC_ROOT.glob('sub-P*')
                   if (d / 'Epoche_selezionate_in_automatico').is_dir())
print(f'{len(subj_dirs)} soggetti con epoche pronte')

def subj_int(name):           # 'sub-P0022'->22 ; 'sub-P000'->0 ; 'sub-P0040'->40
    return int(name.replace('sub-P', ''))

OVERWRITE = False
_SETPAT = re.compile(r'^S(\d+)_epoca_(.+?)_(img|read)$')

counts = defaultdict(int); ranges = []
for sd in tqdm(subj_dirs, desc='soggetti'):
    sid  = subj_int(sd.name)
    base = sd / 'Epoche_selezionate_in_automatico'
    for word in WORDS:
        wdir = base / word
        if not wdir.is_dir():
            continue
        for cond in CONDS:
            for setf in sorted(wdir.glob(f'S*_epoca_{word}_{cond}.set')):
                m = _SETPAT.match(setf.stem)
                if not m:
                    continue
                sess = int(m.group(1))
                try:
                    ep = mne.io.read_epochs_eeglab(str(setf), verbose='ERROR')
                except Exception as e:
                    print('skip', setf.name, '->', e); continue
                X = ep.get_data() * TO_MICROVOLT          # (n_trial, ch, samp) in uV
                outdir = OUT_ROOT / f'P{sid:03d}_S{sess:03d}'
                outdir.mkdir(parents=True, exist_ok=True)
                for k in range(X.shape[0]):
                    x = X[k]
                    if x.shape != (N_CHAN, N_SAMP):
                        counts['bad_shape'] += 1; continue
                    fout = outdir / f'{word}_{cond}_{k:02d}.csv'
                    if OVERWRITE or not fout.exists():
                        pd.DataFrame(x.astype(np.float32)).to_csv(fout, header=False, index=False)
                    counts[cond] += 1
                if len(ranges) < 5:
                    ranges.append((setf.name, float(np.abs(X).mean())))

print('\nScritti:', dict(counts))
print('Ampiezza media |uV| (primi file):')
for n, r in ranges:
    print(f'  {n}: {r:.2f} uV')


## Sanity check

In [ ]:
# Sanity: ricarica un CSV scritto e verifica shape + scala
subdirs = sorted(OUT_ROOT.glob('P*_S*'))
print(f'Sessioni scritte: {len(subdirs)}')
print(f'Soggetti unici  : {len({d.name.split("_")[0] for d in subdirs})}')
ex = next(iter(sorted(OUT_ROOT.glob("P*_S*/*_img_*.csv"))))
arr = pd.read_csv(ex, header=None).values
print(f'Esempio {ex.relative_to(OUT_ROOT)}: shape={arr.shape}, range=[{arr.min():.1f},{arr.max():.1f}] uV')

# conteggio trial img per parola (su tutto il dataset 5words)
from collections import Counter
cnt = Counter()
for f in OUT_ROOT.glob('P*_S*/*_img_*.csv'):
    cnt[f.name.split('_img_')[0]] += 1
print('Trial img per parola:', dict(cnt))
print('Totale trial img    :', sum(cnt.values()))
